# Testlauf 2: A100 – Rot auf Blau

**192 geplante Episoden: 32 unterschiedliche Startlagen × 6 Modelle.** Vier Lagen je Richtung-/Abstandsgruppe; jedes Modell erhält dieselben Positionen und Seeds. Keine Bewegungsaufgabe. Dory ist wegen des noch nicht abgesicherten Fine-Tuning-Sonderpfads ausgeschlossen.

Laufzeit → Laufzeittyp ändern → **A100** auswählen, dann die Zellen der Reihe nach ausführen. Modell und kopflose Physik laufen vollständig in Colab. Die Arbeit wird pro Episode nach Drive gesichert. Der spätere HF-Upload findet ausschließlich bei dir lokal statt; dieses Notebook fragt keinen HF-Token ab.

Die automatische Physikprüfung liefert Messwerte, keine behauptete Erklärung des Modells. Die anschließende inhaltliche Auswertung und Modellauswahl nimmst du selbst vor. Das ist ein überschaubarer Modellselektionslauf, kein unberührtes Abschlusstestset.

In [ ]:
REPO_REF = "main"  # Wird vor Veröffentlichung auf den geprüften Implementierungsstand festgelegt
MAX_HOURS = 6.0       # Stoppt vor dem nächsten Versuch; ein laufender Versuch darf fertig werden
AUTO_RELEASE_GPU = True  # Erst nach geprüftem Export nach Drive die Colab-VM freigeben
DRIVE_FOLDER = "/content/drive/MyDrive/LARA-Stack-Lab/stack-study-v2-32"
# Modellliste, Szenen, Seeds und Budgets stammen aus dem eingefrorenen Protokoll.
# Änderungen daran gehören in einen neuen Studienordner und einen neuen Testplan.

## A100 prüfen und die festgelegte Umgebung einrichten

Die Python-Pakete der Physik sind für diesen Lauf festgelegt. Die Inferenz verwendet die bereits gesperrten GGUF-Dateien und llama.cpp-Revision. Beim ersten Einrichten kann auf älteren Colab-Systemen ein CUDA-Quellbuild erforderlich sein; dessen Log und Fortschritt werden ausgegeben.

In [ ]:
import os, sys, json, subprocess, signal, time, shutil, hashlib, urllib.request
from pathlib import Path
from datetime import datetime, timezone

def command(args, *, cwd=None, env=None, allowed=(0,)):
    print("$", " ".join(map(str,args)), flush=True)
    process = subprocess.Popen(list(map(str,args)), cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line,end="",flush=True)
        result = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid,signal.SIGINT)
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL)
            process.wait()
        raise
    finally:
        process.stdout.close()
    if result not in allowed:
        raise RuntimeError(f"Befehl endete mit {result}; Ausgabe oben prüfen.")
    return result

gpu_text = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                                  "--format=csv,noheader,nounits"],text=True)
print(gpu_text)
gpu_rows = [row for row in gpu_text.splitlines() if row.strip()]
if len(gpu_rows) != 1 or "A100" not in gpu_rows[0]:
    raise RuntimeError("Für diesen Vergleich genau eine A100-Laufzeit auswählen.")
if not isinstance(MAX_HOURS,(int,float)) or not 0 < MAX_HOURS < float("inf"):
    raise ValueError("MAX_HOURS muss endlich und positiv sein.")

PROJECT=Path("/content/lara-stack-study-v2")
if not (PROJECT/".git").exists():
    PROJECT.mkdir(parents=True,exist_ok=True)
    command(["git","init"],cwd=PROJECT)
    command(["git","remote","add","origin","https://github.com/carlo2004/Simulation-Neura-LARA-5.git"],cwd=PROJECT)
    command(["git","fetch","--depth","1","origin",REPO_REF],cwd=PROJECT)
    command(["git","checkout","--detach","FETCH_HEAD"],cwd=PROJECT)
commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=PROJECT,text=True).strip()
if REPO_REF != "main" and commit != REPO_REF:
    raise RuntimeError("Der vorhandene Checkout ist ein anderer Stand. Neue Colab-Laufzeit verwenden.")
print("Software-Commit:",commit)
bootstrap_python=sys.executable
if sys.version_info[:2] not in ((3,12),(3,13)):
    command([sys.executable,"-m","pip","install","uv"])
    command([sys.executable,"-m","uv","python","install","3.12"])
    bootstrap_python=subprocess.check_output([sys.executable,"-m","uv","python","find","3.12"],text=True).strip()
PYTHON=PROJECT/".venv/bin/python"
ENV=dict(os.environ,OPENBLAS_NUM_THREADS="1",STACKLAB_CUDA_RUNTIME="auto")
if not PYTHON.is_file():
    command([bootstrap_python,"-m","venv",PROJECT/".venv"],cwd=PROJECT)
command([PYTHON,"-m","pip","install","-r","requirements-study.txt"],cwd=PROJECT,env=ENV)
command([bootstrap_python,"bootstrap.py"],cwd=PROJECT,env=ENV)
def lab(*args,allowed=(0,)):
    return command([PYTHON,"-m","stacklab",*args],cwd=PROJECT,env=ENV,allowed=allowed)
lab("--version")

## Drive verbinden, Plan erstellen oder fortsetzen

Die 32 Szenen wurden bereits unabhängig physisch geprüft. Ihre Prüfdaten, Pakete und Geometrie-Hashes werden abgeglichen; es werden hier keine zusätzlichen Kontrollsimulationen oder Bewegungsversuche gestartet. In diesem Ordner bleiben es maximal 192 Modell-Episoden. Technische und unterbrochene Versuche werden erhalten und **nicht automatisch wiederholt**.

Nach einer neuen Colab-Sitzung diesen Ablauf mit demselben Drive-Ordner wiederholen. Vorhandene Episoden werden aus geprüften Einzelarchiven wiederhergestellt. Neue Versuche bekommen neue Batch-Manifeste mit der tatsächlich verwendeten GPU, ohne alte Manifeste umzuschreiben. Immer nur eine Colab-Sitzung für denselben Studienordner verwenden.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CHECKPOINT=Path(DRIVE_FOLDER)
WORK=Path("/content/stack-study-v2-data")
lab("experiment","stack-study","prepare","--output",WORK,"--checkpoint-dir",CHECKPOINT)
STUDY=json.loads((WORK/"study.json").read_text(encoding="utf-8"))
SETTINGS=STUDY["settings"]
MODELS=SETTINGS["models"]
print("Protokoll:",STUDY["protocol_sha256"])
print("Modelle:",MODELS)
print("Geplante Episoden:",len(STUDY["cases"])*len(MODELS))
print(json.loads((WORK/"collection-status.json").read_text(encoding="utf-8")))

## Nur die sechs ausgewählten Modelle installieren

Alle nutzen Q4_K_M und denselben Kontext von 16.384 Tokens. Herstellerprofile und Tokenizer bleiben als sichtbare Bedingungen gespeichert. Bis zu vier Programmblöcke pro Episode bekommen echte Ausführungsrückmeldungen. Modell-Fertigmeldungen ersetzen nicht die Kontakt- und Stabilitätsprüfung.

In [ ]:
recipe = "import json,sys; from stacklab.llm.catalog import lock; print(json.dumps([lock(m,device='cuda').get('cuda_runtime_kind') for m in sys.argv[1:]]))"
kinds=json.loads(subprocess.check_output([PYTHON,"-c",recipe,*MODELS],cwd=PROJECT,env=ENV,text=True))
if "source" in kinds:
    if not (PYTHON.parent/"cmake").is_file() and not shutil.which("cmake"):
        command([PYTHON,"-m","pip","install","cmake"],cwd=PROJECT,env=ENV)
    if not shutil.which("g++") or not shutil.which("make"):
        command(["apt-get","update","-qq"])
        command(["apt-get","install","-y","-qq","g++","make"])
    if not shutil.which("nvcc",path=ENV.get("PATH")):
        if not Path("/usr/local/cuda/bin/nvcc").is_file():
            raise RuntimeError("CUDA Toolkit fehlt; eine aktuelle Colab-GPU-Laufzeit verwenden.")
        ENV["PATH"]="/usr/local/cuda/bin:"+ENV.get("PATH","")
for model in MODELS:
    lab("setup","--model",model,"--device","cuda")

## Sammeln, geprüft nach Drive exportieren und GPU freigeben

Der Server bleibt innerhalb dieser Zelle geladen; ein separater Worker setzt die Physik für jede Episode neu auf. Vor den gewerteten Versuchen werden alle sechs Modelle einmal geladen und auf tatsächliche GPU-Auslagerung geprüft. Das sind reine Infrastrukturprüfungen ohne simulierte Roboteraufgaben.

Es entsteht keine automatische Rangliste der gesamten Studie. `trials.csv`/`trials.jsonl` enthalten die 192 vorgesehenen Versuchseinheiten; `attempts.jsonl` und `evidence.zip` enthalten auch Fehler und Teilversuche. Nach `MAX_HOURS` wird vor der nächsten Episode gestoppt. Die Laufzeit kann um den letzten Versuch und den Export länger sein. Ein weiteres Ausführen setzt nur ungestartete Episoden fort.

Standardmäßig wird die Colab-VM **erst nach erfolgreich geprüftem Export nach Drive** freigegeben. `AUTO_RELEASE_GPU=False` lässt sie zur Fehlersuche offen. Der HF-Upload erfolgt später von deinem Rechner; in Colab wird nichts auf HF oder GitHub hochgeladen.

In [ ]:
PORT=8765
URL=f"http://127.0.0.1:{PORT}"
server_log=Path("/content/stack-study-server.log")
server=None
export_verified=False
failure=None

def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda:stream.read(1024*1024),b""):
            digest.update(part)
    return digest.hexdigest()

try:
    with server_log.open("w",encoding="utf-8") as log:
        server=subprocess.Popen([str(PYTHON),"-m","stacklab","serve","--device","cuda",
            "--model",MODELS[0],"--threads",str(SETTINGS["threads"]),"--port",str(PORT),
            "--context-tokens",str(SETTINGS["context_tokens"]),"--allow-model-switch","--install-missing"],
            cwd=PROJECT,env=ENV,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        ready=False
        until=time.monotonic()+200
        opener=urllib.request.build_opener(urllib.request.ProxyHandler({}))
        while time.monotonic()<until:
            if server.poll() is not None:
                raise RuntimeError("Modellserver beendet sich: "+server_log.read_text(encoding="utf-8")[-8000:])
            try:
                with opener.open(URL+"/info",timeout=1) as response:
                    info=json.load(response)
                if info.get("hardware",{}).get("offloaded_layers",0)>0:
                    ready=True
                    break
            except OSError:
                pass
            time.sleep(.25)
        if not ready:
            raise RuntimeError("Server nicht bereit; "+str(server_log))
        lab("experiment","stack-study","run",WORK,"--server-url",URL,
            "--checkpoint-dir",CHECKPOINT,"--max-hours",str(MAX_HOURS),allowed=(0,1))
except BaseException as error:
    failure=error
    print("Erhebung beendet/unterbrochen:",type(error).__name__,str(error),flush=True)
finally:
    if server is not None and server.poll() is None:
        os.killpg(server.pid,signal.SIGINT)
        try: server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid,signal.SIGKILL)
            server.wait()
    # Export even after an interrupted or technically failed collection.
    lab("experiment","stack-study","status",WORK)
    tag=datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    if server_log.exists():
        startup_record=dict(kind="model-server-startup",scored_model_data=False,
            error=str(failure) if failure else None,log=server_log.read_text(encoding="utf-8",errors="replace"))
        for folder in (WORK/"sessions",CHECKPOINT/"sessions"):
            folder.mkdir(parents=True,exist_ok=True)
            target=folder/("server-startup-"+tag+".json")
            temporary=target.with_suffix(".tmp")
            temporary.write_text(json.dumps(startup_record,ensure_ascii=False,indent=2),encoding="utf-8")
            temporary.replace(target)
    EXPORT=Path("/content")/("stack-study-v2-export-"+tag)
    lab("experiment","stack-study","export",WORK,"--output",EXPORT)
    DEST=CHECKPOINT/"exports"/EXPORT.name
    DEST.mkdir(parents=True,exist_ok=False)
    for source in EXPORT.iterdir():
        target=DEST/source.name
        temporary=target.with_suffix(target.suffix+".tmp")
        shutil.copyfile(source,temporary)
        if file_sha(source)!=file_sha(temporary):
            raise RuntimeError("Drive-Kopie stimmt nicht überein: "+source.name)
        temporary.replace(target)
    snapshot=json.loads((DEST/"snapshot.json").read_text(encoding="utf-8"))
    if file_sha(DEST/"evidence.zip") != snapshot["archive"]["sha256"]:
        raise RuntimeError("Prüfsumme des Drive-Exports stimmt nicht.")
    export_verified=True
    print("Export geprüft:",DEST,flush=True)
    print("Später diesen Exportordner lokal herunterladen und mit deinem HF-Token selbst hochladen.",flush=True)
    print("Inventar:",json.loads((DEST/"collection-status.json").read_text(encoding="utf-8")),flush=True)
    if AUTO_RELEASE_GPU and export_verified:
        print("Daten sind gesichert. Colab-GPU wird jetzt freigegeben.",flush=True)
        from google.colab import runtime
        try:
            runtime.unassign()
        except Exception as error:
            print("Automatische Freigabe fehlgeschlagen; Laufzeit bitte über Colab trennen:",str(error),flush=True)
if failure is not None:
    raise failure

## Danach lokal

Den neuesten Ordner aus `MyDrive/LARA-Stack-Lab/stack-study-v2-32/exports/` herunterladen. `snapshot.json` enthält die Dateiprüfsummen; `evidence.zip` enthält die vollständigen Aufzeichnungen. Für Fortsetzungen bleiben die Einzelcheckpoints im Studienordner erhalten. Ein teilweiser Export zählt fehlende oder technisch nicht auswertbare Episoden ausdrücklich weiter.

Die 20 Stabilitätsproben und mehrere Programmblöcke sind keine zusätzlichen unabhängigen Simulationen. Jeder der 192 Einträge entspricht einer neuen Szene für ein Modell. 32 Szenen pro Modell sind ein erster gepaarter Vergleich; zukünftige Bestätigungs-/Fine-Tuning-Tests brauchen neue, bisher ungenutzte Startlagen.

[Versuchsprotokoll und lokale HF-Upload-Anleitung](https://github.com/carlo2004/Simulation-Neura-LARA-5/blob/main/docs/stack-study-v2.md)